In [1]:
import geobr
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import box
import time, sys, os, math
from pyspark.sql import functions as F

In [2]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Importa métodos e/ou funções 
from spark_utils import get_spark_session, write_data_csv, convert_nc_to_spark_dataframe

In [3]:
# Cria uma conexão Spark 
spark = get_spark_session("Municipio_Shape")

In [4]:
def criar_grade_copernicus(min_lon, min_lat, max_lon, max_lat, step = 0.25):
    print(f"--> Gerando grade alinhada do Copernicus (Resolução: {step}°)...")
    
    # FORÇA O ALINHAMENTO aos múltiplos exatos de 0.25 (ex: -74.00, -73.75, -73.50...)
    start_lon = np.round(min_lon / step) * step
    end_lon   = np.round(max_lon / step) * step
    start_lat = np.round(min_lat / step) * step
    end_lat   = np.round(max_lat / step) * step

    # Cria os eixos perfeitos
    lons = np.round(np.arange(start_lon, end_lon + step, step), 2)
    lats = np.round(np.arange(start_lat, end_lat + step, step), 2)
    
    grid_cells = []
    half = step / 2.0
    
    for lat in lats:
        for lon in lons:
            cell_poly = box(lon - half, lat - half, lon + half, lat + half)
            grid_cells.append({
                'id_geo': f"GRID_{lat:.2f}_{lon:.2f}",
                'latitude_centro': round(float(lat), 2),   # Ex: -33.75
                'longitude_centro': round(float(lon), 2),  # Ex: -53.50 (e NÃO -53.49)
                'geometry': cell_poly
            })
            
    gdf_grid = gpd.GeoDataFrame(grid_cells, crs="EPSG:4326")
    print(f"    Total de células geradas na grade: {len(gdf_grid):,}")
    return gdf_grid

def calcular_pesos_sobreposicao(gdf_grid, gdf_municipios, crs_metrico="EPSG:5880"):
    """
    Calcula a área de sobreposição entre a grade climática e os municípios em m².
    EPSG:5880 = SIRGAS 2000 / Brasil Polyconic (Ideal para medições de área no Brasil).
    """
    print("--> Reprojetando geometrias para sistema métrico plano (m²)...")
    grid_proj = gdf_grid.to_crs(crs_metrico)
    mun_proj = gdf_municipios.to_crs(crs_metrico)
    
    # Área total real de cada município do IBGE em m²
    print("--> Calculando áreas totais dos municípios...")
    mun_proj['area_municipio_m2'] = mun_proj.geometry.area
    
    print("--> Executando intersecção geométrica (Overlay Espacial)... isso pode levar de 1 a 3 minutos.")
    start_overlay = time.time()
    
    # Intersecção entre a malha e os municípios
    intersection = gpd.overlay(grid_proj, mun_proj, how='intersection')
    
    end_overlay = time.time()
    print(f"    Intersecção concluída em {end_overlay - start_overlay:.2f} segundos.")
    
    # Área do pedaço (fração) que sobrepõe
    intersection['area_interseccao_m2'] = intersection.geometry.area
    
    print("--> Normalizando os fatores de peso por área...")
    # FATOR DE PESO: Quanto dessa célula representa a área TOTAL do município
    # A soma de 'fator_peso_municipio' para um mesmo município será exatamente 1.0 (100%)
    intersection['fator_peso_municipio'] = (
        intersection['area_interseccao_m2'] / intersection['area_municipio_m2']
    )
    
    # Arredondar para evitar dízimas no Spark
    intersection['fator_peso_municipio'] = intersection['fator_peso_municipio'].round(6)
    
    # Filtrar pequenas ruínas de borda insignificantes (< 0.01% da área do município)
    intersection = intersection[intersection['fator_peso_municipio'] > 0.0001].copy()
    
    # Seleção final de colunas ajustadas para o Modelo Dimensional
    colunas_finais = [
        'id_geo',
        'latitude_centro',
        'longitude_centro',
        'code_muni',           # Código IBGE (ex: 3550308)
        'name_muni',           # Nome do Município (ex: São Paulo)
        'abbrev_state',        # UF (ex: SP)
        'area_municipio_m2',
        'area_interseccao_m2',
        'fator_peso_municipio'
    ]
    
    df_result = pd.DataFrame(intersection[colunas_finais])
    return df_result



In [5]:

tempo_inicio = time.time()
print("=== INICIANDO CONSTRUÇÃO DA LOOKUP TABLE ESPACIAL (IBGE x COPERNICUS) ===")


# 1. Baixar mapa (grade) dos municípios
gdf_ibge = geobr.read_municipality(code_muni="all", year=2022) # 2022 contém os dados mais recentes de acordo com o Censo 



gdf_ibge['geometry'] = gdf_ibge['geometry'].simplify(tolerance=0.005, preserve_topology=True)


# 2. Obter limites e ALINHAR nos múltiplos exatos da Copernicus (0.25)
bounds = gdf_ibge.total_bounds # [min_lon, min_lat, max_lon, max_lat]

# Força o início e o fim a encaixarem exatamente nos quartos de grau (0.00, 0.25, 0.50, 0.75)
step = 0.25
min_lon = math.floor(bounds[0] * 4) / 4
min_lat = math.floor(bounds[1] * 4) / 4
max_lon = math.ceil(bounds[2] * 4) / 4
max_lat = math.ceil(bounds[3] * 4) / 4

print(f"Limites ajustados da grade: Lon [{min_lon} a {max_lon}], Lat [{min_lat} a {max_lat}]")


# 3. Construir a grade estática do Copernicus
print("\n[Passo 2/4] Construindo malha do Copernicus...")
gdf_copernicus = criar_grade_copernicus(
    min_lon=bounds[0] - 0.25,
    min_lat=bounds[1] - 0.25,
    max_lon=bounds[2] + 0.25,
    max_lat=bounds[3] + 0.25,
    step=0.25  # Alterar para 0.1 se seus dados Copernicus forem na resolução de 9 km
)

# 4. Processar a sobreposição de áreas (Overlay)
print("\n[Passo 3/4] Processando Overlay Espacial e Ponderação de Áreas...")
df_lookup = calcular_pesos_sobreposicao(gdf_copernicus, gdf_ibge)

# 5. Salvar arquivo Parquet
print("\n[Passo 4/4] Exportando tabela final...")
nome_arquivo = "d_lookup_grid_municipio_brasil.parquet"
df_lookup.to_parquet(nome_arquivo, index=False)

tempo_total = time.time() - tempo_inicio
print(f"\nCONCLUÍDO COM SUCESSO EM {tempo_total/60:.2f} MINUTOS!")
print(f"--> Tabela gerada: '{nome_arquivo}'")
print(f"--> Total de associações (Célula x Município): {len(df_lookup):,} linhas")

# Exibir amostra dos dados
print("\nAmostra dos dados gerados:")
print(df_lookup.head(10))

=== INICIANDO CONSTRUÇÃO DA LOOKUP TABLE ESPACIAL (IBGE x COPERNICUS) ===
Limites ajustados da grade: Lon [-74.0 a -28.75], Lat [-34.0 a 5.5]

[Passo 2/4] Construindo malha do Copernicus...
--> Gerando grade alinhada do Copernicus (Resolução: 0.25°)...
    Total de células geradas na grade: 29,256

[Passo 3/4] Processando Overlay Espacial e Ponderação de Áreas...
--> Reprojetando geometrias para sistema métrico plano (m²)...
--> Calculando áreas totais dos municípios...
--> Executando intersecção geométrica (Overlay Espacial)... isso pode levar de 1 a 3 minutos.
    Intersecção concluída em 2.40 segundos.
--> Normalizando os fatores de peso por área...

[Passo 4/4] Exportando tabela final...

CONCLUÍDO COM SUCESSO EM 0.12 MINUTOS!
--> Tabela gerada: 'd_lookup_grid_municipio_brasil.parquet'
--> Total de associações (Célula x Município): 34,244 linhas

Amostra dos dados gerados:
               id_geo  latitude_centro  longitude_centro  code_muni  \
0  GRID_-33.75_-53.50           -33.75 

In [8]:
df_grid_municipio_20260814 = spark.read.parquet(r"C:\Marco Conti\Projetos\mais_einstein\Municipio\d_lookup_grid_municipio_brasil-20260814.parquet")
print(df_grid_municipio_20260814.count())
df_grid_municipio = spark.read.parquet(r"C:\Marco Conti\Projetos\mais_einstein\Municipio\d_lookup_grid_municipio_brasil.parquet")
print(df_grid_municipio.count())

# df_grid_municipio.printSchema()

34326
34244


In [10]:
# df_grid_municipio.select("code_muni").dropDuplicates().count() # 34326 Total / 5570 Municípios
df_grid_municipio_20260814.filter("name_muni = 'São Paulo'").show()
df_grid_municipio.filter("name_muni = 'Taboão da Serra'").show()

+------------------+---------------+----------------+---------+---------+------------+--------------------+--------------------+--------------------+
|            id_geo|latitude_centro|longitude_centro|code_muni|name_muni|abbrev_state|   area_municipio_m2| area_interseccao_m2|fator_peso_municipio|
+------------------+---------------+----------------+---------+---------+------------+--------------------+--------------------+--------------------+
|GRID_-24.00_-46.75|          -24.0|          -46.75|3550308.0|São Paulo|          SP|1.5315451331433287E9|2.1411564902671444E8|            0.139804|
|GRID_-24.00_-46.50|          -24.0|           -46.5|3550308.0|São Paulo|          SP|1.5315451331433287E9|1.3755000244662914E7|            0.008981|
|GRID_-23.75_-46.75|         -23.75|          -46.75|3550308.0|São Paulo|          SP|1.5315451331433287E9|4.3983733164520544E8|            0.287185|
|GRID_-23.75_-46.50|         -23.75|           -46.5|3550308.0|São Paulo|          SP|1.531545133143

In [14]:
df_grid_municipio.toPandas().to_file("municipios_.geojson", driver="GeoJSON")

AttributeError: 'DataFrame' object has no attribute 'to_file'